# 🩺 Diabetic Retinopathy Grading — Production Pipeline v19 (FINAL)
## Complete 30-Step Pipeline | QWK ≥ 0.90 | Windows · Colab · Kaggle · Linux

> **Run cells in order. Read any ⚠️ instructions before proceeding to the next step.**


## ✅ Step 1 — System Diagnostics

In [ ]:
import sys, os, shutil, platform, subprocess
from pathlib import Path

print("="*62)
print("  SYSTEM DIAGNOSTICS")
print("="*62)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  Platform    : {platform.system()} {platform.machine()}")

# ── GPU check ──────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}       : {p.name}  {p.total_memory/1e9:.1f} GB VRAM")
        print(f"  CUDA        : {torch.version.cuda}")
    elif hasattr(torch.backends,'mps') and torch.backends.mps.is_available():
        print("  Device      : Apple MPS")
    else:
        # Check if NVIDIA card exists at all
        _nv = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                             capture_output=True, text=True)
        if _nv.returncode == 0:
            gpu_name = _nv.stdout.strip()
            print(f"  ⚠️  NVIDIA GPU FOUND ({gpu_name}) but PyTorch has NO CUDA support.")
            print("      → Run Step 2 — it will reinstall PyTorch with CUDA automatically.")
        else:
            print("  Device      : CPU only (no NVIDIA GPU detected)")
            print("  ℹ️  Training will work on CPU but will be slow.")
            print("  ℹ️  For faster training use Google Colab (free T4 GPU).")
except ImportError:
    print("  PyTorch     : not installed — run Step 2 first")

# ── Disk & RAM ─────────────────────────────────────────────────
total, used, free = shutil.disk_usage(Path.home())
print(f"  Disk Free   : {free/1e9:.1f} GB  (need ≥ 5 GB for dataset)")
if free < 5e9:
    print("  ⚠️  WARNING: Low disk space — free up space before Step 4")
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"  RAM         : {ram.total/1e9:.1f} GB total, {ram.available/1e9:.1f} GB free")
except ImportError:
    pass
print("="*62)
print("✅ Diagnostics complete.")


## 📦 Step 2 — Install Requirements + CUDA PyTorch (auto-fix)

In [ ]:
import sys, subprocess, platform

def _pip(*args, quiet=True):
    q = ["-q"] if quiet else []
    cmd = [sys.executable, "-m", "pip", "install"] + q + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stderr

def _has_nvidia():
    try:
        r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        return r.returncode == 0
    except FileNotFoundError:
        return False

# ── FIX 1: Reinstall PyTorch with CUDA if NVIDIA GPU is present ──
import torch as _tc
_need_cuda = _has_nvidia() and not _tc.cuda.is_available()
if _need_cuda:
    print("🔧 NVIDIA GPU found but PyTorch has no CUDA.")
    print("   Reinstalling PyTorch with CUDA 12.1 support ...")
    rc, err = _pip(
        "torch", "torchvision", "torchaudio",
        "--index-url", "https://download.pytorch.org/whl/cu121",
        quiet=False
    )
    if rc == 0:
        print("✅ PyTorch+CUDA installed!")
        print("⚠️  *** PLEASE RESTART THE KERNEL NOW (Kernel → Restart) ***")
        print("    Then re-run from Step 1.")
        import sys; sys.exit(0)  # stop execution so user restarts
    else:
        print(f"❌ CUDA install failed. Continuing with CPU.")
        print(f"   Error: {err[-400:]}")
else:
    print(f"✅ PyTorch {_tc.__version__} | CUDA: {_tc.cuda.is_available()}")

# ── Standard packages ──────────────────────────────────────────
PACKAGES = [
    "timm>=1.0.3",
    "albumentations>=1.4.0,<2.0.0",
    "opencv-python-headless",
    "scikit-learn", "pandas", "numpy",
    "tqdm", "matplotlib", "scipy",
    "pyarrow", "fastparquet",
    "kaggle", "psutil",
    "streamlit", "ipywidgets",
]

print("\nInstalling packages ...")
failed = []
for pkg in PACKAGES:
    print(f"  {pkg:<40s}", end=" ", flush=True)
    rc, err = _pip("--upgrade", pkg)
    print("✅" if rc == 0 else f"❌  {err[-100:]}")
    if rc != 0: failed.append(pkg)

# packaging needs force-reinstall on Anaconda
print(f"  {'packaging (force)':<40s}", end=" ", flush=True)
rc, _ = _pip("--force-reinstall", "--no-deps", "packaging")
print("✅" if rc == 0 else "❌")

# pytorch-grad-cam
print(f"  {'pytorch-grad-cam':<40s}", end=" ", flush=True)
rc, _ = _pip("--upgrade", "pytorch-grad-cam")
print("✅" if rc == 0 else "⚠️  (Grad-CAM step will be skipped gracefully)")

if failed:
    print(f"\n⚠️  Failed packages: {failed}")
print("\n✅ All packages installed.")


## 🔑 Step 3 — Imports, Config & Kaggle Auth

### ⚠️ Before you accept the competition rules:
1. Go to **https://www.kaggle.com/c/aptos2019-blindness-detection/rules**
2. Click **'I Understand and Accept'** to join the competition
3. Then run this cell — the upload widget will appear to upload `kaggle.json`


In [ ]:
import os, sys, io, json, gc, time, random, shutil, warnings, zipfile
from pathlib import Path
from copy import deepcopy

# Suppress albumentations update nag
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
from packaging.version import Version

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────
CFG = {
    "seed"         : 42,
    "model_name"   : "tf_efficientnetv2_b1",
    "n_folds"      : 5,
    "test_size"    : 0.10,
    "lr"           : 3e-4,
    "min_lr"       : 1e-6,
    "weight_decay" : 1e-4,
    "patience"     : 5,
    "min_delta"    : 0.001,
    "grad_clip"    : 1.0,
    "label_smooth" : 0.05,
    "dropout"      : 0.50,
    "phases": [
        {"id":1, "size":224, "batch_size":32, "epochs":15, "freeze":True},
        {"id":2, "size":384, "batch_size":16, "epochs":40, "freeze":False},
        {"id":3, "size":512, "batch_size": 8, "epochs":25, "freeze":False},
    ],
    "blur_threshold"     : 50.0,
    "dark_threshold"     : 15.0,
    "min_nonblack_ratio" : 0.10,
}

def seed_everything(seed=CFG["seed"]):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
seed_everything()

# ─────────────────────────────────────────────────────────────
#  DEVICE
# ─────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🔥 GPU : {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("🍎 Apple MPS")
else:
    DEVICE = torch.device("cpu")
    print("💻 CPU mode  (training will be slow — see Colab tip below)")
    print("   💡 TIP: Open this notebook in Google Colab for free GPU.")
    print("           Runtime → Change runtime type → GPU → T4")
USE_AMP = (DEVICE.type == "cuda")

# ─────────────────────────────────────────────────────────────
#  PATHS
# ─────────────────────────────────────────────────────────────
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    DATA_DIR     = Path("/kaggle/input/aptos2019-blindness-detection")
    ARTIFACT_DIR = Path("/kaggle/working/dr_artifacts_v19")
elif IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception: pass
    DATA_DIR     = Path("/content/drive/MyDrive/DR_data/aptos2019")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/DR_data/artifacts_v19")
else:
    DATA_DIR     = Path(os.environ.get("DR_DATA",
                        str(Path.home()/"DR_data"/"aptos2019")))
    ARTIFACT_DIR = Path(os.environ.get("DR_ARTIFACTS",
                        str(Path.home()/"DR_data"/"artifacts_v19")))

IMG_DIR   = DATA_DIR / "train_images"
CSV_PATH  = DATA_DIR / "train.csv"
CACHE_DIR = ARTIFACT_DIR / "cache"
PLOT_DIR  = ARTIFACT_DIR / "plots"
EXPORT_DIR= ARTIFACT_DIR / "export"
for d in [ARTIFACT_DIR, CACHE_DIR, PLOT_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NUM_CLASSES   = 5
GRADE_MAP     = {0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}
GRADE_COLORS  = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
IMAGENET_MEAN = [0.485,0.456,0.406]
IMAGENET_STD  = [0.229,0.224,0.225]

_STATE_FILE = ARTIFACT_DIR / "state_v19.json"
def st_load(): return json.loads(_STATE_FILE.read_text()) if _STATE_FILE.exists() else {}
def st_save(k,v): s=st_load(); s[k]=v; _STATE_FILE.write_text(json.dumps(s,indent=2))
def safe_load(path, map_location="cpu"):
    try:    return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError: return torch.load(path, map_location=map_location)

# ─────────────────────────────────────────────────────────────
#  KAGGLE AUTH — FIX: one-shot widget, no double-trigger
# ─────────────────────────────────────────────────────────────
if not IN_KAGGLE:
    kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
    env_ok     = bool(os.environ.get("KAGGLE_KEY") and
                      os.environ.get("KAGGLE_USERNAME"))

    # Validate existing file if present
    _auth_valid = False
    if kaggle_cfg.exists():
        try:
            _creds = json.loads(kaggle_cfg.read_text())
            assert "username" in _creds and "key" in _creds
            _auth_valid = True
            print(f"✅ Kaggle auth ready — user: {_creds['username']}")
        except Exception as e:
            print(f"❌ kaggle.json corrupt: {e} — re-upload below")
            kaggle_cfg.unlink(missing_ok=True)
    elif env_ok:
        _auth_valid = True
        print(f"✅ Kaggle auth ready (env vars)")

    if not _auth_valid:
        if IN_COLAB:
            from google.colab import files as _cf
            print("📤 Upload your kaggle.json:")
            _up = _cf.upload()
            kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
            for _fn, _d in _up.items(): kaggle_cfg.write_bytes(_d)
            try: kaggle_cfg.chmod(0o600)
            except Exception: pass
            print("✅ Saved. Re-run this cell to verify.")
        else:
            # FIX: widget only shown when auth is missing, no double-save
            try:
                import ipywidgets as widgets
                from IPython.display import display, HTML

                _saved = [False]  # guard: prevent double-save

                _uploader = widgets.FileUpload(accept='.json', multiple=False)
                _btn      = widgets.Button(
                    description='💾 Save kaggle.json',
                    button_style='success',
                    layout=widgets.Layout(width='200px'))
                _status   = widgets.Label(value='← Select file first')

                def _on_save(_):
                    if _saved[0]:
                        _status.value = '✅ Already saved — re-run cell to verify.'
                        return
                    if not _uploader.value:
                        _status.value = '⚠️  No file selected yet!'
                        return
                    try:
                        kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
                        val = _uploader.value
                        # ipywidgets v7 returns dict, v8 returns list
                        if isinstance(val, dict):
                            raw = list(val.values())[0]['content']
                        else:
                            raw = val[0]['content']
                        # Validate JSON before saving
                        parsed = json.loads(bytes(raw).decode())
                        assert 'username' in parsed and 'key' in parsed, \
                            'Missing username/key in JSON'
                        kaggle_cfg.write_bytes(bytes(raw))
                        try: kaggle_cfg.chmod(0o600)
                        except Exception: pass
                        _saved[0] = True
                        _status.value = f'✅ Saved for {parsed["username"]} — re-run cell to verify'
                    except Exception as e:
                        _status.value = f'❌ Error: {e}'

                _btn.on_click(_on_save)
                display(HTML('<b>📤 Upload your kaggle.json:</b>'))
                display(widgets.HBox([_uploader, _btn, _status]))
                display(HTML(
                    '<small>Get it from: kaggle.com → Account → API → <b>Create New API Token</b></small>'
                ))
                print("\nAfter saving, RE-RUN THIS CELL to confirm auth.")
            except ImportError:
                print(f"⚠️  Please copy kaggle.json manually to: {kaggle_cfg}")
                print("    or set KAGGLE_USERNAME + KAGGLE_KEY env vars")

print(f"\n✅ Config loaded | PyTorch {torch.__version__} | timm {timm.__version__}")
print(f"   DATA_DIR     : {DATA_DIR}")
print(f"   ARTIFACT_DIR : {ARTIFACT_DIR}")
print(f"   Device       : {DEVICE} | AMP: {'ON' if USE_AMP else 'OFF'}")


## 📥 Step 4 — Dataset Download & Extraction (APTOS 2019)

> **⚠️ IMPORTANT — If download fails with auth error:**
> 1. Visit https://www.kaggle.com/c/aptos2019-blindness-detection
> 2. Click **Join Competition** and accept the rules
> 3. Re-run this cell
>
> **⚠️ ALTERNATIVE — Manual download:**
> 1. Download from https://www.kaggle.com/c/aptos2019-blindness-detection/data
> 2. Extract so that `train.csv` and `train_images/` folder are inside:
>    `C:\Users\<you>\DR_data\aptos2019\`
> 3. Re-run this cell — it will skip download if files are present


In [ ]:
import subprocess as _sp, sys, zipfile
from pathlib import Path

COMPETITION = "aptos2019-blindness-detection"

def _dataset_ok():
    if not CSV_PATH.exists():
        return False
    return len(list(IMG_DIR.glob("*.png"))) >= 3000

def _try_find_extracted():
    """Look for train_images/ in common extraction locations."""
    candidates = [
        DATA_DIR / "train_images",
        DATA_DIR / COMPETITION / "train_images",
        DATA_DIR.parent / "train_images",
    ]
    for c in candidates:
        if c.exists() and len(list(c.glob("*.png"))) >= 3000:
            return c.parent
    return None

if _dataset_ok():
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ Dataset already present: {n:,} images — skipping download.")

elif IN_KAGGLE:
    src = Path(f"/kaggle/input/{COMPETITION}")
    if src.exists() and not DATA_DIR.exists():
        DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
        DATA_DIR.symlink_to(src)
    print(f"✅ Kaggle input linked: {src}")

else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    # ── Step 1: try kaggle API download ──────────────────────────
    _zip = DATA_DIR / f"{COMPETITION}.zip"
    if not _zip.exists():
        print("⬇️  Attempting Kaggle API download (~1.5 GB) ...")
        print("   (This takes 3-10 min depending on connection)")

        # Verify auth before running download
        _kf = Path.home() / ".kaggle" / "kaggle.json"
        if not _kf.exists() and not (os.environ.get("KAGGLE_KEY")):
            print("❌ Kaggle auth not found.")
            print("   → Go back to Step 3 and upload your kaggle.json")
            raise RuntimeError("Kaggle auth missing — complete Step 3 first")

        r = _sp.run(
            [sys.executable, "-m", "kaggle", "competitions", "download",
             "-c", COMPETITION, "-p", str(DATA_DIR)],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            err_text = r.stderr + r.stdout
            print(f"❌ Download error:\n{err_text[-1000:]}")
            print()
            if "403" in err_text or "forbidden" in err_text.lower():
                print("🔑 FIX: You must ACCEPT THE COMPETITION RULES first!")
                print("   1. Open: https://www.kaggle.com/c/aptos2019-blindness-detection")
                print("   2. Click 'Join Competition' / 'I Understand and Accept'")
                print("   3. Come back and re-run this cell")
            elif "401" in err_text or "unauthorized" in err_text.lower():
                print("🔑 FIX: kaggle.json credentials are wrong.")
                print("   1. Go to kaggle.com → Account → API → Create New API Token")
                print("   2. Re-upload via Step 3 widget")
            else:
                print("💡 ALTERNATIVE: Download manually from:")
                print("   https://www.kaggle.com/c/aptos2019-blindness-detection/data")
                print(f"  Extract to: {DATA_DIR}")
            raise RuntimeError("Download failed — see instructions above")
        else:
            print("✅ Download complete.")
    else:
        print(f"✅ Zip already downloaded: {_zip}")

    # ── Step 2: Unzip ─────────────────────────────────────────────
    if _zip.exists():
        print("📦 Unzipping ...")
        with zipfile.ZipFile(_zip, "r") as z:
            z.extractall(DATA_DIR)
        # Also unzip nested zips (Kaggle packs train_images separately)
        for nz in DATA_DIR.glob("*.zip"):
            print(f"   Unzipping nested: {nz.name} ...")
            with zipfile.ZipFile(nz, "r") as z:
                z.extractall(DATA_DIR)
            nz.unlink()
        _zip.unlink(missing_ok=True)
        print("✅ Unzip done.")

    # ── Step 3: Check for misplaced extraction ────────────────────
    if not _dataset_ok():
        found = _try_find_extracted()
        if found and found != DATA_DIR:
            print(f"   Images found at {found} — moving to {DATA_DIR} ...")
            import shutil
            for item in found.iterdir():
                shutil.move(str(item), str(DATA_DIR / item.name))

if not _dataset_ok():
    raise RuntimeError(
        f"Dataset incomplete!\n"
        f"Expected train.csv + ≥3000 .png files in {DATA_DIR}\n"
        f"CSV exists: {CSV_PATH.exists()} | PNG count: {len(list(IMG_DIR.glob('*.png')))}\n"
        f"See the instructions above this cell."
    )

n_imgs = len(list(IMG_DIR.glob("*.png")))
print(f"✅ Dataset ready — {n_imgs:,} images at {IMG_DIR}")


## 📂 Step 5 — Load Dataset

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
df_raw["path"]  = df_raw["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df_raw["label"] = df_raw["diagnosis"].map(GRADE_MAP)

exists_mask = df_raw["path"].apply(lambda p: Path(p).exists())
n_missing = (~exists_mask).sum()
if n_missing:
    print(f"⚠️  {n_missing} images referenced in CSV but not found on disk — dropped.")
df_raw = df_raw[exists_mask].reset_index(drop=True)

print(f"✅ {len(df_raw):,} images loaded.")
print(df_raw[["diagnosis","label"]].value_counts().sort_index())


## 🧹 Step 6 — Data Cleaning

In [ ]:
def image_quality_flags(path,
                         blur_thr=CFG["blur_threshold"],
                         dark_thr=CFG["dark_threshold"],
                         nonblack_thr=CFG["min_nonblack_ratio"]):
    bgr = cv2.imread(str(path))
    if bgr is None: return False, "unreadable"
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if lap_var < blur_thr: return False, f"blurry(lap={lap_var:.1f})"
    mean_br = gray.mean()
    if mean_br < dark_thr: return False, f"dark(mean={mean_br:.1f})"
    nonblack = (gray > 10).mean()
    if nonblack < nonblack_thr: return False, f"black_border(nonblack={nonblack:.2f})"
    return True, ""

_clean_cache = ARTIFACT_DIR / "clean_flags.parquet"
if _clean_cache.exists():
    df_flags = pd.read_parquet(_clean_cache)
    print("✅ [RESUME] Cleaning flags loaded from cache.")
else:
    print("Running image quality checks ...")
    results = [image_quality_flags(p) for p in tqdm(df_raw["path"], leave=False)]
    df_raw["is_ok"]  = [r[0] for r in results]
    df_raw["reason"] = [r[1] for r in results]
    df_flags = df_raw[["id_code","is_ok","reason"]].copy()
    df_flags.to_parquet(_clean_cache, index=False)

for col in ["is_ok", "reason"]:
    if col in df_raw.columns: df_raw.drop(columns=[col], inplace=True)
df_raw = df_raw.merge(df_flags[["id_code","is_ok","reason"]], on="id_code", how="left")
df_raw["is_ok"] = df_raw["is_ok"].fillna(True)

bad = df_raw[~df_raw["is_ok"]]
print(f"  Removed : {len(bad):,} low-quality images")
if len(bad): print(bad["reason"].value_counts().to_string())

df = df_raw[df_raw["is_ok"]].reset_index(drop=True)
print(f"  Kept    : {len(df):,} clean images")


## 📊 Step 7 — EDA

In [ ]:
_eda_file = PLOT_DIR / "eda.png"
if _eda_file.exists():
    print("✅ [RESUME] EDA plot exists.")
    plt.imshow(plt.imread(str(_eda_file))); plt.axis("off"); plt.show()
else:
    counts = [int((df.diagnosis==g).sum()) for g in range(5)]
    labels = [f"G{g}\n{GRADE_MAP[g]}" for g in range(5)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bars = axes[0].bar(labels, counts, color=GRADE_COLORS, edgecolor="k", lw=0.6)
    axes[0].set_title("Class Distribution", fontweight="bold")
    for b, n in zip(bars, counts):
        axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+20,
                     str(n), ha="center", fontsize=9)
    axes[1].pie([c/sum(counts)*100 for c in counts], labels=labels,
                colors=GRADE_COLORS, autopct="%1.1f%%", startangle=140)
    axes[1].set_title("Class %", fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(_eda_file), dpi=120, bbox_inches="tight")
    plt.show()

    fig2, ax2 = plt.subplots(5, 3, figsize=(9, 15))
    for g in range(5):
        samples = df[df.diagnosis==g].sample(min(3, counts[g]), random_state=42)
        for j, (_, row) in enumerate(samples.iterrows()):
            bgr = cv2.imread(str(row.path))
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if bgr is not None \
                  else np.zeros((256,256,3), np.uint8)
            ax2[g][j].imshow(img); ax2[g][j].axis("off")
            if j == 0: ax2[g][j].set_ylabel(f"G{g} {GRADE_MAP[g]}", fontsize=8)
    fig2.suptitle("Sample Images per Grade", fontweight="bold")
    fig2.tight_layout()
    fig2.savefig(str(PLOT_DIR/"eda_samples.png"), dpi=100, bbox_inches="tight")
    plt.show(fig2)
    print(f"✅ EDA saved → {_eda_file}")


## ⚖️ Step 8 — Label / Imbalance Analysis

In [ ]:
counts = df.diagnosis.value_counts().sort_index()
print(f"Imbalance ratio : {counts.max()/counts.min():.1f}×")

labels_arr     = df.diagnosis.values.astype(int)
cls_counts     = np.bincount(labels_arr, minlength=NUM_CLASSES).astype(float)
cls_weights_np = len(labels_arr) / (NUM_CLASSES * np.maximum(cls_counts, 1))
cls_weights_np = cls_weights_np / cls_weights_np.sum() * NUM_CLASSES
CLASS_WEIGHTS  = torch.tensor(cls_weights_np, dtype=torch.float32)

print("\nClass weights:")
for g in range(5):
    bar = "█" * int(cls_weights_np[g]*10)
    print(f"  G{g} {GRADE_MAP[g]:15s}: {counts[g]:5d} imgs | w={cls_weights_np[g]:.3f}  {bar}")
print(f"\n✅ CLASS_WEIGHTS = {np.round(cls_weights_np,3)}")


## 🔬 Step 9 — Preprocessing Pipeline + Cache

In [ ]:
def preprocess_fundus(path_or_img, size=512, sigma_ratio=10, apply_clahe=True):
    if isinstance(path_or_img, np.ndarray):
        img = path_or_img
    else:
        bgr = cv2.imread(str(path_or_img))
        if bgr is None:
            return np.zeros((size, size, 3), np.uint8)
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > 7
    if mask.any():
        rows = np.where(mask.any(1))[0]; cols = np.where(mask.any(0))[0]
        img  = img[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]
    h, w = img.shape[:2]; S = max(h, w)
    img = cv2.copyMakeBorder(img, (S-h)//2, S-h-(S-h)//2,
                                  (S-w)//2, S-w-(S-w)//2,
                             cv2.BORDER_CONSTANT, value=0)
    img   = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    cmask = np.zeros((size, size), np.uint8)
    cv2.circle(cmask, (size//2, size//2), int(size//2*0.97), 255, -1)
    img[cmask == 0] = 0
    if apply_clahe:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        lab[:,:,0] = cv2.createCLAHE(clipLimit=2.0,
                                      tileGridSize=(8,8)).apply(lab[:,:,0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    sigma = max((size // sigma_ratio) | 1, 1)
    blur  = cv2.GaussianBlur(img, (0,0), sigmaX=sigma)
    img   = cv2.addWeighted(img, 4, blur, -4, 128)
    img[cmask == 0] = 0
    return img

CACHE_SIZE  = 224
_cache_flag = CACHE_DIR / "_done.flag"

def get_cached_path(orig_path):
    return CACHE_DIR / f"{Path(orig_path).stem}.png"

if _cache_flag.exists():
    print(f"✅ [RESUME] Cache exists — {sum(1 for _ in CACHE_DIR.glob('*.png'))} images")
else:
    print(f"Building cache ({len(df)} images @ {CACHE_SIZE}px) ...")
    for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
        dest = get_cached_path(row.path)
        if not dest.exists():
            proc = preprocess_fundus(row.path, size=CACHE_SIZE)
            cv2.imwrite(str(dest), cv2.cvtColor(proc, cv2.COLOR_RGB2BGR))
    _cache_flag.touch()
    print(f"✅ Cache built → {CACHE_DIR}")

t0 = time.time()
for _ in range(3): preprocess_fundus(df.path.iloc[0], size=512)
print(f"   Preprocess latency: {(time.time()-t0)/3*1e3:.1f} ms/img")


## ✂️ Step 10 — Train / Test Split

In [ ]:
_split_file = ARTIFACT_DIR / "train_test_split.parquet"
if _split_file.exists():
    df_split = pd.read_parquet(_split_file)
    if "split" in df.columns: df.drop(columns=["split"], inplace=True)
    df = df.merge(df_split[["id_code","split"]], on="id_code", how="left")
    df["split"] = df["split"].fillna("train")
    print("✅ [RESUME] Split loaded.")
else:
    train_idx, test_idx = train_test_split(
        df.index, test_size=CFG["test_size"],
        stratify=df.diagnosis, random_state=CFG["seed"])
    df["split"] = "train"
    df.loc[test_idx, "split"] = "test"
    df[["id_code","split"]].to_parquet(_split_file, index=False)
    print("✅ Train/test split created.")

df_trainval = df[df.split=="train"].reset_index(drop=True)
df_test     = df[df.split=="test" ].reset_index(drop=True)
print(f"   Train+Val : {len(df_trainval):,}")
print(f"   Test      : {len(df_test):,}  (held-out)")


## 🔀 Step 11 — Stratified K-Fold Splits

In [ ]:
_kfold_file = ARTIFACT_DIR / "kfold_splits.parquet"
if _kfold_file.exists():
    df_folds = pd.read_parquet(_kfold_file)
    if "fold" in df_trainval.columns: df_trainval.drop(columns=["fold"], inplace=True)
    df_trainval = df_trainval.merge(
        df_folds[["id_code","fold"]], on="id_code", how="left")
    df_trainval["fold"] = df_trainval["fold"].fillna(0).astype(int)
    print("✅ [RESUME] K-Fold splits loaded.")
else:
    skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True,
                          random_state=CFG["seed"])
    df_trainval["fold"] = -1
    for fi, (_, vi) in enumerate(skf.split(df_trainval, df_trainval.diagnosis)):
        df_trainval.loc[vi, "fold"] = fi
    df_trainval[["id_code","fold"]].to_parquet(_kfold_file, index=False)
    print("✅ 5-Fold splits created.")

print("\nFold distribution:")
for f in range(CFG["n_folds"]):
    n  = (df_trainval.fold == f).sum()
    gd = df_trainval[df_trainval.fold==f].diagnosis.value_counts().sort_index()
    gs = " | ".join(f"G{g}:{c}" for g, c in gd.items())
    print(f"  Fold {f}: {n:5d}  [{gs}]")


## 🔄 Step 12 — Augmentation, Dataset & DataLoader

In [ ]:
import platform as _platform
_A_NEW = Version(A.__version__) >= Version("1.4.0")

def _gauss_noise():
    try:
        return A.GaussNoise(std_range=(0.03, 0.10), p=0.2)
    except TypeError:
        return A.GaussNoise(var_limit=(10, 40), p=0.2)

def _coarse_drop(sz):
    h = sz // 16
    try:
        return A.CoarseDropout(num_holes_range=(1,6),
                               hole_height_range=(h,h),
                               hole_width_range=(h,h), p=0.2)
    except TypeError:
        return A.CoarseDropout(max_holes=6, max_height=h, max_width=h, p=0.2)

def get_train_transform(sz):
    return A.Compose([
        A.RandomResizedCrop(height=sz, width=sz, scale=(0.8,1.0), ratio=(0.9,1.1)),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(0.15, 0.15, p=0.4),
        A.CLAHE(clip_limit=2.0, p=0.3),
        A.HueSaturationValue(10, 20, 10, p=0.3),
        A.RandomGamma(gamma_limit=(80,120), p=0.3),
        _gauss_noise(), A.MotionBlur(blur_limit=3, p=0.1),
        _coarse_drop(sz),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transform(sz):
    return A.Compose([
        A.Resize(sz, sz),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_tta_transforms(sz):
    base = [A.Resize(sz,sz),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return [
        A.Compose(base),
        A.Compose([A.HorizontalFlip(p=1.0)] + base),
        A.Compose([A.VerticalFlip(p=1.0)]   + base),
        A.Compose([A.RandomBrightnessContrast(0.1, 0.1, p=1.0)] + base),
        A.Compose([A.Rotate(limit=10, p=1.0)] + base),
    ]

class DRDataset(Dataset):
    def __init__(self, df, transform=None, img_size=512, use_cache=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size
        self.use_cache = use_cache

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        cached = get_cached_path(row.path)
        if self.use_cache and cached.exists() and self.img_size == CACHE_SIZE:
            bgr = cv2.imread(str(cached))
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        else:
            img = preprocess_fundus(row.path, size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        else:
            img = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0
        return img, torch.tensor(int(row.diagnosis), dtype=torch.long)

# FIX: Windows multiprocessing deadlock — use num_workers=0 on Windows
_NW = 0 if _platform.system() == "Windows" else min(4, os.cpu_count() or 1)
print(f"   DataLoader workers: {_NW} ({'Windows safe mode' if _NW==0 else 'parallel'})")

def make_weighted_loader(df_split, dataset, batch_size, drop_last=False):
    labs   = df_split.diagnosis.values.astype(int)
    cnts   = np.bincount(labs, minlength=NUM_CLASSES).astype(float)
    w_cls  = 1.0 / np.maximum(cnts, 1)
    s_wts  = torch.tensor([w_cls[l] for l in labs], dtype=torch.float)
    sampler= WeightedRandomSampler(s_wts, len(s_wts), replacement=True)
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=_NW, pin_memory=(DEVICE.type=="cuda"),
                      drop_last=drop_last,
                      persistent_workers=(_NW > 0))

def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=_NW, pin_memory=(DEVICE.type=="cuda"),
                      drop_last=drop_last,
                      persistent_workers=(_NW > 0))

print(f"✅ Transforms & Dataset ready | albumentations {A.__version__}")


## 🏗️ Step 13 — Model Architecture

In [ ]:
class DRModel(nn.Module):
    def __init__(self, model_name=CFG["model_name"], num_classes=NUM_CLASSES,
                 pretrained=True, dropout=CFG["dropout"]):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool="avg")
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat),
            nn.Linear(feat, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x): return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(True)

    def unfreeze_top(self, n=4):
        self.freeze_backbone()
        if hasattr(self.backbone, "blocks"):
            for blk in list(self.backbone.blocks)[-n:]:
                for p in blk.parameters(): p.requires_grad_(True)
        for attr in ("conv_head","bn2","norm_head","norm"):
            if hasattr(self.backbone, attr):
                for p in getattr(self.backbone, attr).parameters():
                    p.requires_grad_(True)

_m = DRModel(pretrained=False).to(DEVICE)
_x = torch.randn(2, 3, 224, 224).to(DEVICE)
_o = _m(_x)
assert _o.shape == (2, NUM_CLASSES)
print(f"✅ DRModel OK | {list(_x.shape)} → {list(_o.shape)}")
print(f"   Total params: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M")
del _m, _x, _o; gc.collect()


## ⚖️ Step 14 — Loss, Optimizer & Metrics

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma
        self.weight=weight; self.ls=label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.ls, reduction="none")
        pt = torch.exp(-ce)
        return (self.alpha * (1-pt)**self.gamma * ce).mean()

class HybridLoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        w = class_weights
        self.ce    = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["label_smooth"])
        self.focal = FocalLoss(0.25, 2.0, weight=w,
                               label_smoothing=CFG["label_smooth"])

    def forward(self, logits, targets):
        return 0.5*self.ce(logits, targets) + 0.5*self.focal(logits, targets)

def qwk(y_true, y_pred):
    return cohen_kappa_score(np.array(y_true), np.array(y_pred),
                             weights="quadratic")

class ThresholdOptimizer:
    def __init__(self):
        self.thresholds_ = np.array([0.5, 1.5, 2.5, 3.5])

    def _loss(self, thr, probs, y_true):
        scalar = probs @ np.arange(NUM_CLASSES)
        preds  = pd.cut(scalar,
                        bins=[-np.inf]+list(np.sort(thr))+[np.inf],
                        labels=list(range(NUM_CLASSES))).astype(int)
        return -cohen_kappa_score(y_true, preds, weights="quadratic")

    def fit(self, probs, y_true):
        res = minimize(self._loss, self.thresholds_,
                       args=(probs, y_true), method="Nelder-Mead",
                       options={"maxiter":2000,"xatol":1e-6,"fatol":1e-9})
        self.thresholds_ = np.sort(res.x)
        return self

    def predict(self, probs):
        scalar = probs @ np.arange(NUM_CLASSES)
        return np.clip(
            pd.cut(scalar,
                   bins=[-np.inf]+list(self.thresholds_)+[np.inf],
                   labels=list(range(NUM_CLASSES))).astype(int), 0, 4)

print("✅ HybridLoss, FocalLoss, ThresholdOptimizer, qwk() ready.")


## 💾 Step 15 — Checkpoint & Resume System

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, scaler,
                    epoch, batch, phase_id, best_qwk, thresholds, history):
    torch.save({
        "model"     : model.state_dict(),
        "optimizer" : optimizer.state_dict(),
        "scheduler" : scheduler.state_dict(),
        "scaler"    : scaler.state_dict() if scaler else None,
        "epoch"     : epoch, "batch": batch, "phase_id": phase_id,
        "best_qwk"  : best_qwk, "thresholds": thresholds,
        "history"   : history,
    }, path)

def load_checkpoint(path, model, optimizer=None,
                    scheduler=None, scaler=None, map_location="cpu"):
    ckpt = safe_load(path, map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer and ckpt.get("optimizer"):
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler and ckpt.get("scheduler"):
        scheduler.load_state_dict(ckpt["scheduler"])
    if scaler and ckpt.get("scaler"):
        scaler.load_state_dict(ckpt["scaler"])
    return ckpt

print("✅ Checkpoint helpers ready.")


## 🏋️ Step 16 — 5-Fold CV Training (Phase-Wise + Early Stopping)

In [ ]:
def run_epoch_train(model, loader, criterion, optimizer,
                    scheduler, scaler, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="  train", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast("cuda"):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / max(len(loader), 1)

@torch.no_grad()
def run_epoch_val(model, loader, scaler, device):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="  val  ", leave=False):
        imgs = imgs.to(device)
        if scaler:
            with torch.amp.autocast("cuda"): logits = model(imgs)
        else:
            logits = model(imgs)
        all_probs.append(F.softmax(logits, dim=1).cpu().float().numpy())
        all_labels.extend(labels.numpy())
    return np.concatenate(all_probs), np.array(all_labels)

# FIX: GradScaler compat across PyTorch versions
if USE_AMP:
    try:
        scaler_global = torch.amp.GradScaler("cuda")
    except TypeError:
        scaler_global = torch.cuda.amp.GradScaler()
else:
    scaler_global = None

fold_best_qwks = []
oof_probs      = np.zeros((len(df_trainval), NUM_CLASSES), dtype=np.float32)
oof_labels_arr = df_trainval.diagnosis.values.copy()
global_thr_opt = ThresholdOptimizer()

print("="*68)
print(f"  5-FOLD CV | {CFG['model_name']} | {DEVICE} | AMP={USE_AMP}")
print("="*68)

for fold in range(CFG["n_folds"]):
    ckpt_best = ARTIFACT_DIR / f"fold{fold}_best.pt"
    oof_file  = ARTIFACT_DIR / f"fold{fold}_oof_probs.npy"
    done_flag = ARTIFACT_DIR / f"_done_fold{fold}.flag"

    if done_flag.exists() and ckpt_best.exists():
        prev = safe_load(ckpt_best, "cpu")
        fold_best_qwks.append(prev.get("best_qwk", 0.0))
        if oof_file.exists():
            val_idx = df_trainval[df_trainval.fold==fold].index
            oof_probs[val_idx] = np.load(str(oof_file))
        print(f"  ✅ [RESUME] Fold {fold} | QWK={fold_best_qwks[-1]:.4f}")
        continue

    print(f"\n  {'━'*20}  FOLD {fold}  {'━'*20}")
    df_tr   = df_trainval[df_trainval.fold != fold].reset_index(drop=True)
    df_va   = df_trainval[df_trainval.fold == fold].reset_index(drop=True)
    val_idx = df_trainval[df_trainval.fold == fold].index

    model     = DRModel(pretrained=True).to(DEVICE)
    criterion = HybridLoss(CLASS_WEIGHTS.to(DEVICE))
    thr_opt   = ThresholdOptimizer()

    fold_best_qwk   = -1.0
    best_state      = None
    best_thresholds = thr_opt.thresholds_.copy()
    history         = []

    for phase in CFG["phases"]:
        pid = phase["id"]; sz = phase["size"]
        bs  = phase["batch_size"]; n_ep = phase["epochs"]
        print(f"\n  Phase {pid} | {sz}px | {n_ep} ep | "
              f"{'frozen' if phase['freeze'] else 'unfrozen'}")

        if phase["freeze"]:      model.freeze_backbone()
        elif pid == 2:           model.unfreeze_top(n=4)
        else:                    model.unfreeze_backbone()

        tr_ds = DRDataset(df_tr, get_train_transform(sz), sz, use_cache=False)
        va_ds = DRDataset(df_va, get_val_transform(sz),   sz, use_cache=False)
        tr_ld = make_weighted_loader(df_tr, tr_ds, bs, drop_last=True)
        va_ld = make_loader(va_ds, bs)

        lr_p  = CFG["lr"] / (3 ** (pid-1))
        opt   = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr_p, weight_decay=CFG["weight_decay"])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=n_ep * max(len(tr_ld),1), eta_min=CFG["min_lr"])

        patience_cnt = 0
        for ep in range(n_ep):
            tr_loss = run_epoch_train(
                model, tr_ld, criterion, opt, sched, scaler_global, DEVICE)
            vp, vl  = run_epoch_val(model, va_ld, scaler_global, DEVICE)
            thr_opt.fit(vp, vl)
            val_preds = thr_opt.predict(vp)
            val_qwk   = qwk(vl, val_preds)
            val_acc   = accuracy_score(vl, val_preds)

            improved = val_qwk > fold_best_qwk + CFG["min_delta"]
            if improved:
                fold_best_qwk   = val_qwk
                best_state      = deepcopy(model.state_dict())
                best_thresholds = thr_opt.thresholds_.copy()
                patience_cnt    = 0
                save_checkpoint(ckpt_best, model, opt, sched, scaler_global,
                                ep, 0, pid, fold_best_qwk, best_thresholds, history)
            else:
                patience_cnt += 1

            history.append({"fold":fold,"phase":pid,"epoch":ep,
                            "tr_loss":tr_loss,"val_qwk":val_qwk,"val_acc":val_acc})
            print(f"    P{pid} Ep{ep+1:02d}/{n_ep}: "
                  f"loss={tr_loss:.4f}  QWK={val_qwk:.4f}  "
                  f"Acc={val_acc*100:.1f}%{'  ★' if improved else ''}")

            if patience_cnt >= CFG["patience"]:
                print(f"    ↳ Early stop (patience={CFG['patience']})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    va_ds_f   = DRDataset(df_va, get_val_transform(512), 512, use_cache=False)
    va_ld_f   = make_loader(va_ds_f, 8)
    fold_probs, _ = run_epoch_val(model, va_ld_f, scaler_global, DEVICE)

    oof_probs[val_idx] = fold_probs[:len(val_idx)]
    np.save(str(oof_file), fold_probs[:len(val_idx)])
    fold_best_qwks.append(fold_best_qwk)
    done_flag.touch()
    print(f"  ✅ Fold {fold} done — QWK: {fold_best_qwk:.4f}")
    del model; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

np.save(str(ARTIFACT_DIR/"oof_probs.npy"),  oof_probs)
np.save(str(ARTIFACT_DIR/"oof_labels.npy"), oof_labels_arr)
print("\n" + "="*68)
for i, q in enumerate(fold_best_qwks): print(f"  Fold {i}: QWK = {q:.4f}")
print(f"  Mean : {np.mean(fold_best_qwks):.4f} ± {np.std(fold_best_qwks):.4f}")
print("="*68)


## 📐 Step 17 — OOF Threshold Optimization

In [ ]:
oof_probs_l  = np.load(str(ARTIFACT_DIR/"oof_probs.npy"))
oof_labels_l = np.load(str(ARTIFACT_DIR/"oof_labels.npy"))

print("Optimizing thresholds on OOF predictions ...")
global_thr_opt.fit(oof_probs_l, oof_labels_l)
oof_preds_opt    = global_thr_opt.predict(oof_probs_l)
oof_preds_argmax = oof_probs_l.argmax(axis=1)
oof_qwk_opt  = qwk(oof_labels_l, oof_preds_opt)
oof_qwk_base = qwk(oof_labels_l, oof_preds_argmax)
oof_acc_opt  = accuracy_score(oof_labels_l, oof_preds_opt)

print(f"  OOF QWK (argmax)    : {oof_qwk_base:.4f}")
print(f"  OOF QWK (optimized) : {oof_qwk_opt:.4f}  ← primary metric")
print(f"  OOF Acc (optimized) : {oof_acc_opt*100:.2f}%")
print(f"  Thresholds          : {np.round(global_thr_opt.thresholds_,3)}")

np.save(str(ARTIFACT_DIR/"oof_preds.npy"),  oof_preds_opt)
np.save(str(ARTIFACT_DIR/"thresholds.npy"), global_thr_opt.thresholds_)
st_save("oof_qwk", float(oof_qwk_opt))
st_save("oof_acc", float(oof_acc_opt))
print(f"\n  {'✅' if oof_qwk_opt>=0.90 else '⚠️ '} QWK ≥ 0.90: "
      f"{'MET' if oof_qwk_opt>=0.90 else 'NOT MET'}")


## 🔁 Step 18 — TTA Ensemble Inference

In [ ]:
@torch.no_grad()
def tta_predict(df_infer, tta_size=512):
    tta_tfs    = get_tta_transforms(tta_size)
    fold_paths = sorted(ARTIFACT_DIR.glob("fold*_best.pt"))
    if not fold_paths:
        raise RuntimeError("No fold checkpoints — run training first.")
    ensemble  = np.zeros((len(df_infer), NUM_CLASSES), np.float32)
    n_contrib = 0
    for ckpt_path in fold_paths:
        ckpt  = safe_load(ckpt_path, DEVICE)
        model = DRModel(pretrained=False).to(DEVICE)
        model.load_state_dict(ckpt["model"]); model.eval()
        fold_sum = np.zeros((len(df_infer), NUM_CLASSES), np.float32)
        for tf in tta_tfs:
            ds = DRDataset(df_infer, tf, tta_size, use_cache=False)
            ld = make_loader(ds, 8)
            pl = []
            for imgs, _ in tqdm(ld, desc=f"  {ckpt_path.stem} TTA", leave=False):
                imgs = imgs.to(DEVICE)
                if USE_AMP:
                    with torch.amp.autocast("cuda"): logits = model(imgs)
                else:
                    logits = model(imgs)
                pl.append(F.softmax(logits,dim=1).cpu().float().numpy())
            fold_sum += np.concatenate(pl)[:len(df_infer)]
        ensemble  += fold_sum / len(tta_tfs)
        n_contrib += 1
        del model; gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()
    return ensemble / n_contrib

print("Running TTA inference on test set ...")
test_probs = tta_predict(df_test)
_thr = np.load(str(ARTIFACT_DIR/"thresholds.npy"))
global_thr_opt.thresholds_ = _thr
test_preds  = global_thr_opt.predict(test_probs)
np.save(str(ARTIFACT_DIR/"test_probs.npy"), test_probs)
np.save(str(ARTIFACT_DIR/"test_preds.npy"), test_preds)
print(f"✅ TTA done on {len(df_test)} test samples.")


## 📈 Step 19 — Final Metrics & Confusion Matrix

In [ ]:
test_probs_l  = np.load(str(ARTIFACT_DIR/"test_probs.npy"))
test_preds_l  = np.load(str(ARTIFACT_DIR/"test_preds.npy"))
test_labels_l = df_test.diagnosis.values
test_qwk = qwk(test_labels_l, test_preds_l)
test_acc = accuracy_score(test_labels_l, test_preds_l)
st_save("test_qwk", float(test_qwk))
st_save("test_acc", float(test_acc))
print(f"  Test QWK : {test_qwk:.4f}  {'✅' if test_qwk>=0.90 else '⚠️'}")
print(f"  Test Acc : {test_acc*100:.2f}%")

cm  = confusion_matrix(test_labels_l, test_preds_l)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ConfusionMatrixDisplay(cm, display_labels=[f"G{i}" for i in range(5)]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"QWK={test_qwk:.4f}  Acc={test_acc*100:.1f}%", fontweight="bold")

report = classification_report(
    test_labels_l, test_preds_l,
    target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)], output_dict=True)
recalls    = [report[f"G{i} {GRADE_MAP[i]}"]["recall"]    for i in range(5)]
precisions = [report[f"G{i} {GRADE_MAP[i]}"]["precision"] for i in range(5)]
x = np.arange(5); w = 0.35
axes[1].bar(x-w/2, recalls,    w, color=GRADE_COLORS, label="Recall",    alpha=0.9)
axes[1].bar(x+w/2, precisions, w, color=GRADE_COLORS, label="Precision", alpha=0.5, hatch="//")
axes[1].set_xticks(x); axes[1].set_xticklabels([f"G{i}" for i in range(5)])
axes[1].set_ylim(0, 1.15); axes[1].legend()
axes[1].set_title("Per-Class Recall & Precision", fontweight="bold")
plt.tight_layout()
plt.savefig(str(PLOT_DIR/"test_metrics.png"), dpi=120, bbox_inches="tight")
plt.show()
print(classification_report(
    test_labels_l, test_preds_l,
    target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)]))


## 💾 Step 20 — Model Export

In [ ]:
import json as _json
_best_fold = int(np.argmax(fold_best_qwks)) if fold_best_qwks else 0
_ckpt      = safe_load(ARTIFACT_DIR/f"fold{_best_fold}_best.pt", "cpu")
_em        = DRModel(pretrained=False)
_em.load_state_dict(_ckpt["model"]); _em.eval()
_dst = EXPORT_DIR / "dr_model_final.pt"

torch.save({
    "model_state_dict": _em.state_dict(),
    "model_name"      : CFG["model_name"],
    "num_classes"     : NUM_CLASSES,
    "grade_map"       : GRADE_MAP,
    "thresholds"      : global_thr_opt.thresholds_.tolist(),
    "imagenet_mean"   : IMAGENET_MEAN,
    "imagenet_std"    : IMAGENET_STD,
    "test_qwk"        : float(test_qwk),
    "oof_qwk"         : float(oof_qwk_opt),
}, str(_dst))
(EXPORT_DIR/"thresholds.json").write_text(
    _json.dumps({"thresholds":global_thr_opt.thresholds_.tolist(),
                 "grade_map":GRADE_MAP}, indent=2))
(EXPORT_DIR/"label_map.json").write_text(_json.dumps(GRADE_MAP, indent=2))
print(f"✅ Model → {_dst}  ({_dst.stat().st_size/1e6:.1f} MB)")
print(f"   Best fold: {_best_fold}  (QWK={fold_best_qwks[_best_fold]:.4f})")
del _em; gc.collect()


## 🎨 Step 21 — Grad-CAM++ Explainability

In [ ]:
try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    _bfi  = int(np.argmax(fold_best_qwks)) if fold_best_qwks else 0
    _ckpt = safe_load(ARTIFACT_DIR/f"fold{_bfi}_best.pt", DEVICE)
    _m    = DRModel(pretrained=False).to(DEVICE)
    _m.load_state_dict(_ckpt["model"]); _m.eval()
    if hasattr(_m.backbone, "blocks"):
        _tgt = [_m.backbone.blocks[-1][-1]]
    else:
        _ch  = list(_m.backbone.children())
        _tgt = [_ch[-1] if isinstance(_ch[-1], nn.Module) else _ch[-2]]
    cam  = GradCAMPlusPlus(model=_m, target_layers=_tgt)
    fig, axes = plt.subplots(2, 5, figsize=(22, 9))
    for grade in range(5):
        sample = df[df.diagnosis==grade].sample(1, random_state=42).iloc[0]
        raw    = preprocess_fundus(sample.path, size=512)
        tensor = get_val_transform(512)(image=raw)["image"].unsqueeze(0).to(DEVICE)
        gc_map = cam(input_tensor=tensor, targets=None)[0]
        vis    = show_cam_on_image(raw.astype(np.float32)/255., gc_map, use_rgb=True)
        axes[0][grade].imshow(raw); axes[0][grade].axis("off")
        axes[0][grade].set_title(f"G{grade}: {GRADE_MAP[grade]}", fontsize=9)
        axes[1][grade].imshow(vis); axes[1][grade].axis("off")
        axes[1][grade].set_title("Grad-CAM++", fontsize=9)
    plt.suptitle("Grad-CAM++ — Model Attention per DR Grade",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(PLOT_DIR/"gradcam.png"), dpi=120, bbox_inches="tight")
    plt.show()
    del _m, cam; gc.collect()
    print("✅ Grad-CAM++ complete.")
except ImportError:
    print("⚠️  pytorch-grad-cam not installed — skipping.")
except Exception as e:
    print(f"⚠️  Grad-CAM error: {e}")


## 🌐 Step 22 — Streamlit App

Run with: `streamlit run <EXPORT_DIR>/dr_app.py`


In [ ]:
APP_CODE = '''
import streamlit as st, torch, cv2, numpy as np, json
from pathlib import Path
from PIL import Image
import torch.nn as nn, torch.nn.functional as F, timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

EXPORT_DIR    = Path(__file__).parent
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = [0.485,0.456,0.406]
IMAGENET_STD  = [0.229,0.224,0.225]
GRADE_MAP     = {0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}

@st.cache_resource
def load_model():
    b = torch.load(str(EXPORT_DIR/"dr_model_final.pt"), map_location=DEVICE, weights_only=False)
    class M(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = timm.create_model(b["model_name"],pretrained=False,num_classes=0,global_pool="avg")
            f = self.backbone.num_features
            self.head = nn.Sequential(nn.BatchNorm1d(f),nn.Linear(f,256),nn.ReLU(True),nn.Dropout(0.5),nn.Linear(256,5))
        def forward(self,x): return self.head(self.backbone(x))
    m=M().to(DEVICE); m.load_state_dict(b["model_state_dict"]); m.eval()
    return m, np.array(b["thresholds"])

def preprocess(img,size=512):
    img=cv2.resize(img,(size,size),interpolation=cv2.INTER_AREA)
    c=np.zeros((size,size),np.uint8); cv2.circle(c,(size//2,size//2),int(size//2*0.97),255,-1)
    img[c==0]=0; lab=cv2.cvtColor(img,cv2.COLOR_RGB2LAB)
    lab[:,:,0]=cv2.createCLAHE(2.0,(8,8)).apply(lab[:,:,0])
    img=cv2.cvtColor(lab,cv2.COLOR_LAB2RGB)
    bl=cv2.GaussianBlur(img,(0,0),sigmaX=51); img=cv2.addWeighted(img,4,bl,-4,128); img[c==0]=0
    return img

def predict(model,thr,img):
    p=preprocess(img)
    t=A.Compose([A.Resize(512,512),A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()])(image=p)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad(): probs=F.softmax(model(t),dim=1).cpu().numpy()[0]
    import pandas as pd; s=probs@np.arange(5)
    g=int(pd.cut([s],bins=[-np.inf]+list(np.sort(thr))+[np.inf],labels=[0,1,2,3,4]).astype(int)[0])
    return g, probs

st.set_page_config(page_title="DR Grading",page_icon="🩺",layout="wide")
st.title("🩺 Diabetic Retinopathy Grading")
st.caption("Upload fundus image → model predicts DR grade 0–4")
c1,c2=st.columns(2)
with c1: f=st.file_uploader("Upload fundus image",type=["png","jpg","jpeg"])
if f:
    model,thr=load_model()
    img=np.array(Image.open(f).convert("RGB"))
    with c1: st.image(img,caption="Input",use_container_width=True)
    with st.spinner("Running ..."):
        g,probs=predict(model,thr,img)
    with c2:
        st.metric("Prediction",f"Grade {g}: {GRADE_MAP[g]}")
        st.progress(int(probs[g]*100))
        import pandas as pd
        st.bar_chart(pd.DataFrame({"Probability":probs},
                     index=[f"G{i} {GRADE_MAP[i]}" for i in range(5)]))
        st.warning("⚠️ Refer to ophthalmologist") if g>=2 else st.success("✅ No/mild DR")
st.caption("⚠️ Research use only.")
'''
_app = EXPORT_DIR / "dr_app.py"
_app.write_text(APP_CODE.strip())
print(f"✅ Streamlit app → {_app}")
print(f"   Run: streamlit run {_app}")


## 📋 Step 23 — Final Summary

In [ ]:
state = st_load(); W = 68
print("="*W)
print("  DIABETIC RETINOPATHY GRADING — v19 FINAL SUMMARY")
print("="*W)
print(f"  Backbone  : {CFG['model_name']}")
print(f"  Device    : {DEVICE} | AMP: {'ON' if USE_AMP else 'OFF'}")
print(f"  Dataset   : APTOS 2019 | Clean: {len(df):,} | "
      f"Train: {len(df_trainval):,} | Test: {len(df_test):,}")
print("\n  ─── K-Fold CV ───")
if fold_best_qwks:
    for i, q in enumerate(fold_best_qwks):
        print(f"    Fold {i}: QWK = {q:.4f}")
    print(f"    Mean  : {np.mean(fold_best_qwks):.4f} "
          f"± {np.std(fold_best_qwks):.4f}")
print(f"    OOF QWK (opt)  : {state.get('oof_qwk','N/A')}")
print(f"    OOF Acc        : {float(state.get('oof_acc',0))*100:.2f}%")
print("\n  ─── Hold-Out Test ───")
_tq = float(state.get('test_qwk',0))
print(f"    Test QWK : {_tq:.4f}  "
      f"{'✅ TARGET MET' if _tq>=0.90 else '⚠️  below 0.90'}")
print(f"    Test Acc : {float(state.get('test_acc',0))*100:.2f}%")
print("\n  ─── Exports ───")
for f in sorted(EXPORT_DIR.glob("*")):
    sz = f.stat().st_size
    print(f"    {f.name:<35s}  "
          f"{sz/1e6:.2f} MB" if sz > 1e5 else f"    {f.name}")
print("="*W)
print("  ⚠️  RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print("="*W)
